In [2]:
with open('paratodos.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [6]:
# read poetry txt file and inspect its size
print("dataset size:", len(text))

dataset size: 34990


In [7]:
# first 500 chars
print(text[:500])

Para todos

Revisão: Karina Miotto
Para todos

Daniel Cukier

Para ela

Puro presente

O tempo acabou
Ela se foi, eu me fui
E nós não fomos
É como se o vento que já nem
Soprava tão bem 
Decidisse parar de soprar
Vou assim. E sobrarão os
Pensamentos de nós dois
De mim por ela e de ela por mim
Sonhos às vezes, um cheiro ou outro
Aqui e ali, talvez chorando
Talvez sorrindo em lembrar da
Saudade, em lembrar daquele sorriso.

Se um dia eu voltar, nada será
Como foi, ela não estará mais aqui,
Nem aqui


In [8]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

	
 !"'(),-./0123456789:?@ABCDEFGHIJKLMNOPQRSTUVXYabcdefghijklmnopqrstuvwxzÀÉÍàáâãçéêíóôõúü–
91


In [10]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # from string to list of ints
decode = lambda l: ''.join([itos[i] for i in l]) # from list of ints to string

print(encode("Isso é legal"))
print(decode(encode("Isso é legal")))

[33, 67, 67, 63, 2, 82, 2, 60, 53, 55, 49, 60]
Isso é legal


In [11]:
# encode all text and store it in a tensor
import torch 
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:500])

torch.Size([34990]) torch.int64
tensor([40, 49, 66, 49,  2, 68, 63, 52, 63, 67,  1,  1, 42, 53, 70, 57, 67, 80,
        63, 22,  2, 35, 49, 66, 57, 62, 49,  2, 37, 57, 63, 68, 68, 63,  1, 40,
        49, 66, 49,  2, 68, 63, 52, 63, 67,  1,  1, 28, 49, 62, 57, 53, 60,  2,
        27, 69, 59, 57, 53, 66,  1,  1, 40, 49, 66, 49,  2, 53, 60, 49,  1,  1,
        40, 69, 66, 63,  2, 64, 66, 53, 67, 53, 62, 68, 53,  1,  1, 39,  2, 68,
        53, 61, 64, 63,  2, 49, 51, 49, 50, 63, 69,  1, 29, 60, 49,  2, 67, 53,
         2, 54, 63, 57,  8,  2, 53, 69,  2, 61, 53,  2, 54, 69, 57,  1, 29,  2,
        62, 85, 67,  2, 62, 80, 63,  2, 54, 63, 61, 63, 67,  1, 75,  2, 51, 63,
        61, 63,  2, 67, 53,  2, 63,  2, 70, 53, 62, 68, 63,  2, 65, 69, 53,  2,
        58, 78,  2, 62, 53, 61,  1, 43, 63, 64, 66, 49, 70, 49,  2, 68, 80, 63,
         2, 50, 53, 61,  2,  1, 28, 53, 51, 57, 52, 57, 67, 67, 53,  2, 64, 49,
        66, 49, 66,  2, 52, 53,  2, 67, 63, 64, 66, 49, 66,  1, 46, 63, 69,  2,
        

In [12]:
# separate train and validation data
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [13]:
block_size = 8
train_data[:block_size+1]

tensor([40, 49, 66, 49,  2, 68, 63, 52, 63])

In [14]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"input is {context}, target: {target}")

input is tensor([40]), target: 49
input is tensor([40, 49]), target: 66
input is tensor([40, 49, 66]), target: 49
input is tensor([40, 49, 66, 49]), target: 2
input is tensor([40, 49, 66, 49,  2]), target: 68
input is tensor([40, 49, 66, 49,  2, 68]), target: 63
input is tensor([40, 49, 66, 49,  2, 68, 63]), target: 52
input is tensor([40, 49, 66, 49,  2, 68, 63, 52]), target: 63


In [31]:
torch.manual_seed(1337)
batch_size = 4 # how many parallel sequences we process
block_size = 8 # context length for predictions

def get_batch(split):
    # small batch of data of inputs and targets
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----------------')
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"input: {context.tolist()} target: {target}")


inputs:
torch.Size([4, 8])
tensor([[53, 61,  2, 54, 57, 61,  1, 39],
        [49,  2, 49,  2, 51, 49, 62, 81],
        [52, 63, 66, 49,  1, 30, 53, 51],
        [ 2, 52, 53,  2, 49, 61, 63, 66]])
targets:
torch.Size([4, 8])
tensor([[61,  2, 54, 57, 61,  1, 39, 51],
        [ 2, 49,  2, 51, 49, 62, 81, 80],
        [63, 66, 49,  1, 30, 53, 51, 56],
        [52, 53,  2, 49, 61, 63, 66,  1]])
----------------
input: [53] target: 61
input: [53, 61] target: 2
input: [53, 61, 2] target: 54
input: [53, 61, 2, 54] target: 57
input: [53, 61, 2, 54, 57] target: 61
input: [53, 61, 2, 54, 57, 61] target: 1
input: [53, 61, 2, 54, 57, 61, 1] target: 39
input: [53, 61, 2, 54, 57, 61, 1, 39] target: 51
input: [49] target: 2
input: [49, 2] target: 49
input: [49, 2, 49] target: 2
input: [49, 2, 49, 2] target: 51
input: [49, 2, 49, 2, 51] target: 49
input: [49, 2, 49, 2, 51, 49] target: 62
input: [49, 2, 49, 2, 51, 49, 62] target: 81
input: [49, 2, 49, 2, 51, 49, 62, 81] target: 80
input: [52] target: 63

In [34]:
print(xb)

tensor([[53, 61,  2, 54, 57, 61,  1, 39],
        [49,  2, 49,  2, 51, 49, 62, 81],
        [52, 63, 66, 49,  1, 30, 53, 51],
        [ 2, 52, 53,  2, 49, 61, 63, 66]])


In [60]:
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx (B, T)
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :] # (B, C)
            probs = F.softmax(logits, dim=-1) # (B,C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
            
            

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)

print(logits.shape)
print(f'Initial loss: {loss.item()}')

logits, loss = m(xb, None)
print(logits.shape)


idx = torch.zeros((1,1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

batch_size = 32
print('tranining 10k steps...')
for steps in range(10000):
    xb, yb = get_batch('train')

    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print(f'Loss after training: {loss.item()}')
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))


torch.Size([256, 91])
Initial loss: 5.10022497177124
torch.Size([32, 8, 91])
	iáTXá6x1w?8nôzçGfQw"owJ C,k0tCUMrRóuMP8áôMôKekGííCLfr/	iá68SD(QNStüáiBxóérõqt	RKzfé1(jêCUI3HqçkéT-v9
tranining 10k steps...
Loss after training: 2.322233200073242
	
Pa minharcou misto es tire emade vorrazende erio s polonh sináquisba fého
Tare cto fe arasRelas
E P


In [64]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [74]:
# version 1
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev, 0)

In [83]:
# version 2
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ---> (B, T, C)
torch.allclose(xbow, xbow2)

True

In [84]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

True

In [95]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32
x = torch.randn(B,T,C)

#let's see a single head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)

wei = q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T,T))
# wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v

# out = wei @ x
out.shape

torch.Size([4, 8, 16])

In [96]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [75]:
torch.tril(torch.ones(3,3))

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [69]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [77]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print("a=")
print(a)
print("b=")
print(b)
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
